# Complete Human Pose & Movement Analysis with MediaPipe

This notebook builds a **general-purpose** system for analyzing human pose from images and video.

The project is not limited to a single joint (e.g. only the knee). It can analyze a whole set of body joints:

- Knee
- Hip
- Ankle
- Shoulder
- Elbow
- Wrist, to the extent MediaPipe Pose landmarks allow

### Overall Pipeline

```
Image / Video → OpenCV → Frame → MediaPipe Pose Landmarker → Pose Landmarks
→ Landmark Coordinates → Joint Angles → Visualization → Frame-by-Frame Data
→ Pandas DataFrame → CSV → Graphs → Movement Analysis → Annotated Video
```

### The Most Important Rule

This notebook must be fully runnable from top to bottom. If you **Restart** the kernel and then
click **Run All**, it must work without errors and without depending on any prior cell execution
order. No variable is ever used before it is defined.

> ⚠️ **Important warning**: The output of this project is a computer-vision analysis of movement,
> **not** a medical diagnostic tool. Angle accuracy depends heavily on factors such as camera
> angle, lighting, and image quality (see section 33).

## 2. Environment & Imports

We import the libraries required for the project. We use the MediaPipe **Tasks API**
(not the legacy `mp.solutions.pose` API), since the Tasks API is the officially supported,
more stable interface, and is better suited for Video mode.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("MediaPipe version:", mp.__version__)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## 3. Configuration

Project paths are defined here, in a single cell. Assumed project structure:

```
project/
│
├── pose_analysis.ipynb
├── pose_landmarker_full.task
│
├── videos/
│   └── test.mp4
│
└── output/
```

If your video path is different, just change the value of `VIDEO_PATH` in this same cell.

In [ ]:
# ----- Project paths -----
# These are absolute Windows paths. A raw string r"..." is used so that
# backslashes in the Windows path are interpreted literally.
MODEL_PATH = Path(r"C:\Users\Hosei\Desktop\project\sample_project\anaconda_projects\db\projext\pose_landmarker_full.task")
VIDEO_PATH = Path(r"C:\Users\Hosei\Desktop\project\sample_project\anaconda_projects\db\projext\videos\my_rdl.mp4")
OUTPUT_DIR = Path(r"C:\Users\Hosei\Desktop\project\sample_project\anaconda_projects\db\projext\output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO_PATH = OUTPUT_DIR / "annotated_pose.mp4"
OUTPUT_CSV_PATH = OUTPUT_DIR / "pose_angles.csv"

# ----- Analysis settings -----
VISIBILITY_THRESHOLD = 0.5  # a landmark below this visibility is considered unreliable

print("MODEL_PATH:", MODEL_PATH.resolve())
print("VIDEO_PATH:", VIDEO_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 4. Validate Files

Before running anything in MediaPipe, we check that the model file and the video file
actually exist, so that we get a clear, understandable error instead of an obscure
traceback from deep inside MediaPipe.

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH.resolve()}\n"
        "Download the .task model file and place it next to this notebook."
    )

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video file not found: {VIDEO_PATH.resolve()}\n"
        "Place an actual video file at videos/test.mp4, or change VIDEO_PATH in the Configuration cell."
    )

print("Model and video files found successfully.")

## 5. Create MediaPipe Detector (IMAGE mode)

For the initial exploration on a single frame, we create a detector with `RunningMode.IMAGE`.
Later, in the full-video processing section, a separate detector with `RunningMode.VIDEO`
is created, since these two modes are not interchangeable.

In [ ]:
base_options_image = python.BaseOptions(
    model_asset_path=str(MODEL_PATH)
)

options_image = vision.PoseLandmarkerOptions(
    base_options=base_options_image,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

detector_image = vision.PoseLandmarker.create_from_options(options_image)

print("Image-mode PoseLandmarker created.")

## 6. Read Video Metadata

Before processing, we read the video's basic metadata (FPS, frame count, dimensions).
This is needed both for computing each frame's timestamp and for building the
VideoWriter later on.

In [ ]:
cap_meta = cv2.VideoCapture(str(VIDEO_PATH))

if not cap_meta.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH.resolve()}")

fps = cap_meta.get(cv2.CAP_PROP_FPS)
frame_count = int(cap_meta.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = frame_count / fps if fps > 0 else 0

cap_meta.release()

print("FPS:", fps)
print("Frames:", frame_count)
print("Width:", width)
print("Height:", height)
print("Duration (s):", duration)

## 7. Read First Frame

Before processing the whole video, we first read just a single frame, so we can test the
entire pipeline on one simple example. We never access `frame.shape` without first
checking that the frame was read successfully, since `frame` can be `None` on failure.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH.resolve()}")

success, first_frame = cap.read()

cap.release()

if not success or first_frame is None:
    raise RuntimeError("Could not read the first frame.")

print("Frame shape:", first_frame.shape)

## 8. Display First Frame

To display the image inside Jupyter we use `matplotlib` (not `cv2.imshow()`, which does not
work well in a notebook environment). We build a helper function that resizes only the
*display* copy of the frame; the aspect ratio is always preserved and the image is never
cropped or stretched.

> Note: this resize is for **display only**. The original frame (`first_frame`) remains
> untouched, and that same original frame is what will be used for analysis and saving
> (see section 32).

In [ ]:
def resize_for_display(frame, max_width=800, max_height=900):
    """Returns a resized copy of the frame for display purposes only; aspect ratio is preserved."""
    h, w = frame.shape[:2]

    scale = min(
        max_width / w,
        max_height / h,
        1.0
    )

    new_width = int(w * scale)
    new_height = int(h * scale)

    return cv2.resize(
        frame,
        (new_width, new_height),
        interpolation=cv2.INTER_AREA
    )


display_frame = resize_for_display(first_frame)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("First Frame")
plt.show()

## 9. Detect Pose

OpenCV reads images in **BGR** color space, while MediaPipe expects **RGB**. So we first
convert the color space and then build an `mp.Image`.

After calling `detect()`, **before** accessing `result.pose_landmarks[0]`, we check whether
any pose was actually detected; otherwise this line would fail with an error.

In [ ]:
rgb_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb_frame
)

result = detector_image.detect(mp_image)

if not result.pose_landmarks:
    print("No pose detected.")
    landmarks = None
else:
    print("Pose detected.")
    landmarks = result.pose_landmarks[0]

## 10. Inspect Landmarks

If a pose was detected, `landmarks` is a list of 33 landmarks. Each landmark has the
following attributes:

- `landmark.x`, `landmark.y` — coordinates **normalized** to (0, 1) relative to image width/height
- `landmark.z` — relative depth (section 34)
- `landmark.visibility` — confidence that this point is visible

To convert normalized coordinates to actual image pixels:

In [ ]:
def landmark_to_pixel(landmark, frame):
    """Converts a landmark's normalized (0-1) coordinates into pixel coordinates on frame."""
    h, w = frame.shape[:2]
    x = int(landmark.x * w)
    y = int(landmark.y * h)
    return x, y


if landmarks is not None:
    sample = landmarks[0]
    print("Landmark 0 (Nose):")
    print("  x:", sample.x)
    print("  y:", sample.y)
    print("  z:", sample.z)
    print("  visibility:", sample.visibility)
    print("  pixel:", landmark_to_pixel(sample, first_frame))
else:
    print("Nothing to inspect — no pose was detected in the first frame.")

## 11. Landmark Reference

MediaPipe Pose produces 33 landmarks in total. For this project, the following landmarks
matter most:

| Index | Name |
|---|---|
| 0 | Nose |
| 11 | Left Shoulder |
| 12 | Right Shoulder |
| 13 | Left Elbow |
| 14 | Right Elbow |
| 15 | Left Wrist |
| 16 | Right Wrist |
| 23 | Left Hip |
| 24 | Right Hip |
| 25 | Left Knee |
| 26 | Right Knee |
| 27 | Left Ankle |
| 28 | Right Ankle |
| 29 | Left Heel |
| 30 | Right Heel |
| 31 | Left Foot Index |
| 32 | Right Foot Index |

> **Important**: "Left" and "Right" are relative to the **subject's own body**, not
> necessarily left/right of the image. If a person faces the camera, their "Left Shoulder"
> will appear on the right side of the image.

In [ ]:
POSE_LANDMARKS = {
    "nose": 0,
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_elbow": 13,
    "right_elbow": 14,
    "left_wrist": 15,
    "right_wrist": 16,
    "left_hip": 23,
    "right_hip": 24,
    "left_knee": 25,
    "right_knee": 26,
    "left_ankle": 27,
    "right_ankle": 28,
    "left_heel": 29,
    "right_heel": 30,
    "left_foot_index": 31,
    "right_foot_index": 32,
}

for name, idx in POSE_LANDMARKS.items():
    print(f"{idx:>2}  {name}")

## 12. Coordinate Conversion and Visibility Check

We build two helper functions:

- `get_landmark(landmarks, name)`: gives access to a landmark by name (e.g. `"right_knee"`)
  instead of using a raw index number throughout the code.
- `is_landmark_visible(landmark, threshold)`: checks whether this landmark's `visibility`
  is above the threshold. If visibility is low, the landmark's coordinates may be
  unreliable (e.g. due to occlusion or being out of frame).

In [ ]:
def get_landmark(landmarks, name):
    """Returns the landmark corresponding to a given name (e.g. 'right_knee')."""
    return landmarks[POSE_LANDMARKS[name]]


def is_landmark_visible(landmark, threshold=VISIBILITY_THRESHOLD):
    """Checks whether this landmark is visible with sufficient confidence."""
    return landmark.visibility >= threshold


if landmarks is not None:
    right_knee_lm = get_landmark(landmarks, "right_knee")
    print("Right knee visibility:", right_knee_lm.visibility)

    if not is_landmark_visible(right_knee_lm):
        print("Right knee landmark may be unreliable.")
    else:
        print("Right knee landmark looks reliable.")

## 13. Generic Angle Calculation Function

The heart of this project is a generic function that calculates the angle between three
points A, B, and C, measured precisely at point **B**:

```
A
 \
  \
   B
    \
     \
      C
```

We use the law of cosines (dot product of two vectors):

```
cos(angle) = (BA · BC) / (|BA| * |BC|)
```

Using `np.clip(cosine_angle, -1.0, 1.0)` is essential, because due to floating-point error,
the computed value sometimes slightly exceeds the valid range `[-1, 1]` (e.g.
`1.0000000002`); in that case `np.arccos` would return `NaN`. `clip` fixes this issue at
the root.

If one of the vectors has zero length (i.e. two points coincide exactly), the function
returns `np.nan` instead of crashing.

In [ ]:
def calculate_angle(a, b, c):
    """
    Calculate angle ABC in degrees.
    The angle is measured at point B.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)

    ba = a - b
    bc = c - b

    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)

    if norm_ba == 0 or norm_bc == 0:
        return np.nan

    cosine_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine_angle))
    return float(angle)


# Quick sanity check: right angle (90 degrees)
test_angle = calculate_angle([0, 1], [0, 0], [1, 0])
print("Test angle (should be ~90):", test_angle)

## 14. Knee Angle

The knee angle is measured between three points: **Hip → Knee → Ankle**:

```
Hip
 |
 |
Knee
 |
 |
Ankle
```

In [ ]:
if landmarks is not None:
    right_hip = get_landmark(landmarks, "right_hip")
    right_knee = get_landmark(landmarks, "right_knee")
    right_ankle = get_landmark(landmarks, "right_ankle")

    right_knee_angle = calculate_angle(
        [right_hip.x, right_hip.y],
        [right_knee.x, right_knee.y],
        [right_ankle.x, right_ankle.y]
    )

    print(f"Right knee angle: {right_knee_angle:.1f}°")

    left_hip = get_landmark(landmarks, "left_hip")
    left_knee = get_landmark(landmarks, "left_knee")
    left_ankle = get_landmark(landmarks, "left_ankle")

    left_knee_angle = calculate_angle(
        [left_hip.x, left_hip.y],
        [left_knee.x, left_knee.y],
        [left_ankle.x, left_ankle.y]
    )

    print(f"Left knee angle: {left_knee_angle:.1f}°")

## 15. Hip Angle

The hip angle is measured between **Shoulder → Hip → Knee**:

```
Shoulder
    |
    |
   Hip
    |
    |
   Knee
```

In [ ]:
if landmarks is not None:
    right_shoulder = get_landmark(landmarks, "right_shoulder")

    right_hip_angle = calculate_angle(
        [right_shoulder.x, right_shoulder.y],
        [right_hip.x, right_hip.y],
        [right_knee.x, right_knee.y]
    )
    print(f"Right hip angle: {right_hip_angle:.1f}°")

    left_shoulder = get_landmark(landmarks, "left_shoulder")

    left_hip_angle = calculate_angle(
        [left_shoulder.x, left_shoulder.y],
        [left_hip.x, left_hip.y],
        [left_knee.x, left_knee.y]
    )
    print(f"Left hip angle: {left_hip_angle:.1f}°")

## 16. Ankle Angle

The ankle angle is measured between **Knee → Ankle → Foot Index**:

```
Knee ---- Ankle ---- Foot
```

In [ ]:
if landmarks is not None:
    right_foot_index = get_landmark(landmarks, "right_foot_index")

    right_ankle_angle = calculate_angle(
        [right_knee.x, right_knee.y],
        [right_ankle.x, right_ankle.y],
        [right_foot_index.x, right_foot_index.y]
    )
    print(f"Right ankle angle: {right_ankle_angle:.1f}°")

    left_foot_index = get_landmark(landmarks, "left_foot_index")

    left_ankle_angle = calculate_angle(
        [left_knee.x, left_knee.y],
        [left_ankle.x, left_ankle.y],
        [left_foot_index.x, left_foot_index.y]
    )
    print(f"Left ankle angle: {left_ankle_angle:.1f}°")

## 17. Elbow Angle

The elbow angle is measured between **Shoulder → Elbow → Wrist**:

```
Shoulder ---- Elbow ---- Wrist
```

In [ ]:
if landmarks is not None:
    right_elbow = get_landmark(landmarks, "right_elbow")
    right_wrist = get_landmark(landmarks, "right_wrist")

    right_elbow_angle = calculate_angle(
        [right_shoulder.x, right_shoulder.y],
        [right_elbow.x, right_elbow.y],
        [right_wrist.x, right_wrist.y]
    )
    print(f"Right elbow angle: {right_elbow_angle:.1f}°")

    left_elbow = get_landmark(landmarks, "left_elbow")
    left_wrist = get_landmark(landmarks, "left_wrist")

    left_elbow_angle = calculate_angle(
        [left_shoulder.x, left_shoulder.y],
        [left_elbow.x, left_elbow.y],
        [left_wrist.x, left_wrist.y]
    )
    print(f"Left elbow angle: {left_elbow_angle:.1f}°")

## 18. Shoulder Angle

The shoulder angle is measured between **Hip → Shoulder → Elbow**:

```
Hip ---- Shoulder ---- Elbow
```

In [ ]:
if landmarks is not None:
    right_shoulder_angle = calculate_angle(
        [right_hip.x, right_hip.y],
        [right_shoulder.x, right_shoulder.y],
        [right_elbow.x, right_elbow.y]
    )
    print(f"Right shoulder angle: {right_shoulder_angle:.1f}°")

    left_shoulder_angle = calculate_angle(
        [left_hip.x, left_hip.y],
        [left_shoulder.x, left_shoulder.y],
        [left_elbow.x, left_elbow.y]
    )
    print(f"Left shoulder angle: {left_shoulder_angle:.1f}°")

## 19. A Note on the Wrist

The `Pose Landmarker` is not suitable for detailed hand analysis (e.g. finger angles or
precise wrist flexion), since it only has a single point (Wrist) for the entire hand. For
more detailed finger and hand-joint analysis, a separate model — **MediaPipe Hand
Landmarker** — could be used in the future.

In this version, for the hand we only use the **Elbow → Wrist** vector (which was also
used in computing the elbow angle above).

## 20. The Central `calculate_joint_angles` Function

So far we computed each angle separately and manually, to keep each one's logic clear.
Now we consolidate all of this logic into a single function whose input is the list of 33
landmarks and whose output is a dictionary of all angles. This function will be used in
the full-video processing stage.

If the visibility of one of the three points required for an angle is below the threshold,
that angle becomes `np.nan` instead of an invalid number (so it is not mistakenly treated
as a real measurement in plots and statistics).

In [ ]:
def calculate_joint_angles(landmarks, visibility_threshold=VISIBILITY_THRESHOLD):
    """
    Given the list of landmarks for one frame, returns a dictionary of all joint angles.
    If one of the three landmarks required for an angle is not sufficiently visible,
    that angle will be np.nan instead of a number.
    """

    def safe_angle(name_a, name_b, name_c):
        lm_a = get_landmark(landmarks, name_a)
        lm_b = get_landmark(landmarks, name_b)
        lm_c = get_landmark(landmarks, name_c)

        if not (
            is_landmark_visible(lm_a, visibility_threshold)
            and is_landmark_visible(lm_b, visibility_threshold)
            and is_landmark_visible(lm_c, visibility_threshold)
        ):
            return np.nan

        return calculate_angle(
            [lm_a.x, lm_a.y],
            [lm_b.x, lm_b.y],
            [lm_c.x, lm_c.y]
        )

    return {
        "right_knee_angle": safe_angle("right_hip", "right_knee", "right_ankle"),
        "left_knee_angle": safe_angle("left_hip", "left_knee", "left_ankle"),

        "right_hip_angle": safe_angle("right_shoulder", "right_hip", "right_knee"),
        "left_hip_angle": safe_angle("left_shoulder", "left_hip", "left_knee"),

        "right_ankle_angle": safe_angle("right_knee", "right_ankle", "right_foot_index"),
        "left_ankle_angle": safe_angle("left_knee", "left_ankle", "left_foot_index"),

        "right_elbow_angle": safe_angle("right_shoulder", "right_elbow", "right_wrist"),
        "left_elbow_angle": safe_angle("left_shoulder", "left_elbow", "left_wrist"),

        "right_shoulder_angle": safe_angle("right_hip", "right_shoulder", "right_elbow"),
        "left_shoulder_angle": safe_angle("left_hip", "left_shoulder", "left_elbow"),
    }


if landmarks is not None:
    all_angles = calculate_joint_angles(landmarks)
    for k, v in all_angles.items():
        print(f"{k:>22}: {v:.1f}°" if not np.isnan(v) else f"{k:>22}: NaN")

## 21. Drawing Functions (Landmark / Connection / Angle)

We build three generic functions for drawing on a frame:

- `draw_point`: draws a single landmark (point) on the frame.
- `draw_connection`: draws a line between two landmarks.
- `draw_angle`: writes an angle's numeric value next to a landmark.

In [ ]:
def draw_point(frame, landmark, color, radius=8):
    x, y = landmark_to_pixel(landmark, frame)
    cv2.circle(frame, (x, y), radius, color, -1)


def draw_connection(frame, landmark_a, landmark_b, color, thickness=4):
    x1, y1 = landmark_to_pixel(landmark_a, frame)
    x2, y2 = landmark_to_pixel(landmark_b, frame)
    cv2.line(frame, (x1, y1), (x2, y2), color, thickness)


def draw_angle(frame, landmark, angle, text_offset=(20, -20)):
    if np.isnan(angle):
        return
    x, y = landmark_to_pixel(landmark, frame)
    cv2.putText(
        frame,
        f"{angle:.1f} deg",
        (x + text_offset[0], y + text_offset[1]),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )


print("Drawing helper functions defined: draw_point, draw_connection, draw_angle")

## 22. Full Visualization of the Knee Angle on One Frame

Now we combine all three drawing functions on the first frame, so we can see the
Hip–Knee–Ankle triangle and the right knee angle drawn on the image:

```
Hip ●
     \
      \
       ● Knee   152.9°
        \
         \
          ● Ankle
```

Note: all drawing is done on a **copy** of the original frame, so `first_frame` stays
untouched.

In [ ]:
if landmarks is not None:
    annotated_single = first_frame.copy()

    draw_connection(annotated_single, right_hip, right_knee, (255, 0, 0))
    draw_connection(annotated_single, right_knee, right_ankle, (255, 0, 0))

    draw_point(annotated_single, right_hip, (0, 0, 255))
    draw_point(annotated_single, right_knee, (0, 0, 255))
    draw_point(annotated_single, right_ankle, (0, 0, 255))

    draw_angle(annotated_single, right_knee, right_knee_angle)

    display_annotated = resize_for_display(annotated_single)

    plt.figure(figsize=(8, 10))
    plt.imshow(cv2.cvtColor(display_annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Right Knee Angle Visualization")
    plt.show()
else:
    print("No pose detected in the first frame — nothing to visualize.")

## 23. Processing the Full Video

Now that the pipeline works correctly on a single frame, we move on to the full video.

To process a video we must use `RunningMode.VIDEO` (not `IMAGE`), and for each frame an
**increasing timestamp** based on FPS must be computed and passed to `detect_for_video()`:

```python
timestamp_ms = int(frame_index * 1000 / fps)
result = detector.detect_for_video(mp_image, timestamp_ms)
```

We consolidate all of this logic into a single, comprehensive, reusable function called
`process_video()`.

In [ ]:
# Colors for the joint connections drawn on each video frame
JOINT_CONNECTIONS = [
    # (landmark_a, landmark_b, color BGR)
    ("right_shoulder", "right_elbow", (255, 0, 0)),
    ("right_elbow", "right_wrist", (255, 0, 0)),
    ("left_shoulder", "left_elbow", (0, 255, 255)),
    ("left_elbow", "left_wrist", (0, 255, 255)),

    ("right_hip", "right_knee", (0, 0, 255)),
    ("right_knee", "right_ankle", (0, 0, 255)),
    ("left_hip", "left_knee", (255, 0, 255)),
    ("left_knee", "left_ankle", (255, 0, 255)),

    ("right_shoulder", "right_hip", (0, 255, 0)),
    ("left_shoulder", "left_hip", (0, 255, 0)),
    ("right_shoulder", "left_shoulder", (0, 255, 0)),
    ("right_hip", "left_hip", (0, 255, 0)),

    ("right_knee", "right_foot_index", (0, 128, 255)),
    ("left_knee", "left_foot_index", (0, 128, 255)),
]

# Which landmark each angle label is drawn next to
ANGLE_LABEL_POINT = {
    "right_knee_angle": "right_knee",
    "left_knee_angle": "left_knee",
    "right_hip_angle": "right_hip",
    "left_hip_angle": "left_hip",
    "right_ankle_angle": "right_ankle",
    "left_ankle_angle": "left_ankle",
    "right_elbow_angle": "right_elbow",
    "left_elbow_angle": "left_elbow",
    "right_shoulder_angle": "right_shoulder",
    "left_shoulder_angle": "left_shoulder",
}


def draw_pose_annotations(frame, landmarks, angles):
    """Draws all connections, points, and angle values on frame (in place)."""
    for name_a, name_b, color in JOINT_CONNECTIONS:
        lm_a = get_landmark(landmarks, name_a)
        lm_b = get_landmark(landmarks, name_b)
        draw_connection(frame, lm_a, lm_b, color)

    for name in POSE_LANDMARKS:
        draw_point(frame, get_landmark(landmarks, name), (0, 0, 255), radius=5)

    for angle_name, landmark_name in ANGLE_LABEL_POINT.items():
        angle_value = angles.get(angle_name, np.nan)
        draw_angle(frame, get_landmark(landmarks, landmark_name), angle_value)

    return frame


print("draw_pose_annotations() defined.")

## 24. The Final `process_video` Function

This function runs the entire pipeline from start to finish: opening the video, creating a
detector for Video mode, frame-by-frame processing, angle calculation, optional drawing,
optional annotated video saving, and finally building and saving the DataFrame.

Error handling in this function:
- If the video fails to open → `RuntimeError`
- If a frame can't be read (`None`) → that frame is skipped, the program does not crash
- If no pose is detected in a frame → all angles for that frame are recorded as `NaN`

In [ ]:
def process_video(
    video_path,
    model_path,
    output_csv=None,
    output_video=None,
    show=False,
):
    """
    Full processing of a video: pose detection in every frame, joint-angle computation,
    saving results to CSV, and optionally saving an annotated video.

    Parameters
    ----------
    video_path : Path or str
    model_path : Path or str
    output_csv : Path or None
        If given, the DataFrame is saved to this path.
    output_video : Path or None
        If given, an annotated video (with landmarks and angles drawn) is saved to this path.
    show : bool
        If True, processing progress is printed every so many frames.

    Returns
    -------
    pd.DataFrame
        One row per frame that was successfully read.
    """
    video_path = Path(video_path)
    model_path = Path(model_path)

    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path.resolve()}")
    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path.resolve()}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path.resolve()}")

    local_fps = cap.get(cv2.CAP_PROP_FPS)
    local_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    local_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    local_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if local_fps <= 0:
        local_fps = 30.0  # safe fallback for videos that don't report a valid FPS

    video_options = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=str(model_path)),
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    video_detector = vision.PoseLandmarker.create_from_options(video_options)

    writer = None
    if output_video is not None:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(
            str(output_video), fourcc, local_fps, (local_width, local_height)
        )

    records = []
    frame_index = 0
    last_timestamp_ms = -1

    while True:
        success, frame = cap.read()

        if not success or frame is None:
            break  # end of video or a broken frame — stop processing, don't crash

        timestamp_ms = int(frame_index * 1000 / local_fps)
        if timestamp_ms <= last_timestamp_ms:
            timestamp_ms = last_timestamp_ms + 1  # guarantee the timestamp keeps increasing
        last_timestamp_ms = timestamp_ms

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        frame_result = video_detector.detect_for_video(mp_img, timestamp_ms)

        if frame_result.pose_landmarks:
            frame_landmarks = frame_result.pose_landmarks[0]
            frame_angles = calculate_joint_angles(frame_landmarks)

            if writer is not None:
                draw_pose_annotations(frame, frame_landmarks, frame_angles)
        else:
            frame_angles = {
                "right_knee_angle": np.nan, "left_knee_angle": np.nan,
                "right_hip_angle": np.nan, "left_hip_angle": np.nan,
                "right_ankle_angle": np.nan, "left_ankle_angle": np.nan,
                "right_elbow_angle": np.nan, "left_elbow_angle": np.nan,
                "right_shoulder_angle": np.nan, "left_shoulder_angle": np.nan,
            }

        record = {"frame": frame_index, "timestamp_ms": timestamp_ms}
        record.update(frame_angles)
        records.append(record)

        if writer is not None:
            writer.write(frame)  # original-size frame, not the resized display copy

        if show and frame_index % 30 == 0:
            print(f"Processed frame {frame_index}/{local_frame_count}")

        frame_index += 1

    cap.release()
    if writer is not None:
        writer.release()

    result_df = pd.DataFrame(records)

    if output_csv is not None:
        result_df.to_csv(output_csv, index=False)

    return result_df


print("process_video() defined.")

## 25. Running the Full Video Processing

With a single function call, the entire video is processed: angles are computed, an
annotated video is saved, and the result is both saved as CSV and returned as a
DataFrame.

In [ ]:
df = process_video(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_csv=OUTPUT_CSV_PATH,
    output_video=OUTPUT_VIDEO_PATH,
    show=True,
)

print("Processed frames:", len(df))
df.head()

## 26. Right Knee Angle Over Time

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["timestamp_ms"] / 1000, df["right_knee_angle"])
plt.xlabel("Time (seconds)")
plt.ylabel("Angle (degrees)")
plt.title("Right Knee Angle Over Time")
plt.grid(True)
plt.show()

## 27. Left vs Right Knee Comparison

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["timestamp_ms"] / 1000, df["right_knee_angle"], label="Right Knee")
plt.plot(df["timestamp_ms"] / 1000, df["left_knee_angle"], label="Left Knee")
plt.xlabel("Time (seconds)")
plt.ylabel("Angle (degrees)")
plt.title("Left vs Right Knee")
plt.legend()
plt.grid(True)
plt.show()

## 28. Multi-Joint Plot (Right Side)

By changing the `columns_to_plot` list, any combination of joints can be compared together.

In [ ]:
columns_to_plot = [
    "right_knee_angle",
    "right_hip_angle",
    "right_ankle_angle",
    "right_elbow_angle",
    "right_shoulder_angle",
]

plt.figure(figsize=(12, 6))
for col in columns_to_plot:
    plt.plot(df["timestamp_ms"] / 1000, df[col], label=col)

plt.xlabel("Time (seconds)")
plt.ylabel("Angle (degrees)")
plt.title("Right-Side Joint Angles Over Time")
plt.legend()
plt.grid(True)
plt.show()

## 29. Smoothing (Optional)

Raw angle signals usually have some jitter. A rolling mean can reduce this noise. **The raw
data is never overwritten** — a new column is simply added for the smoothed version.

In [ ]:
df["right_knee_angle_smooth"] = (
    df["right_knee_angle"]
    .rolling(window=5, min_periods=1)
    .mean()
)

plt.figure(figsize=(12, 5))
plt.plot(df["timestamp_ms"] / 1000, df["right_knee_angle"], alpha=0.4, label="Raw")
plt.plot(df["timestamp_ms"] / 1000, df["right_knee_angle_smooth"], label="Smoothed (window=5)")
plt.xlabel("Time (seconds)")
plt.ylabel("Angle (degrees)")
plt.title("Right Knee Angle: Raw vs Smoothed")
plt.legend()
plt.grid(True)
plt.show()

## 30. Basic Movement Analysis

Basic statistics (min, max, average) for the knee angle:

### Example: The Squat Concept

The general pattern of a squat is as follows — without introducing any threshold as a
definitive *medical* value:

```
Standing → Knee angle decreases → Squat position (Minimum angle) → Standing again
```

In [ ]:
print("Minimum:", df["right_knee_angle"].min())
print("Maximum:", df["right_knee_angle"].max())
print("Average:", df["right_knee_angle"].mean())

## 31. Repetition Counting (Optional)

A simple, tunable counter for counting repetitions of a movement (e.g. squats) based on
the knee angle crossing a "down" and an "up" threshold. This is only a simple example and
should be calibrated for different movements and camera setups.

In [ ]:
def count_repetitions(angle_series, down_threshold=100, up_threshold=160):
    """
    Simple repetition counter based on the Standing -> Down -> Up pattern.

    When the angle drops below down_threshold, we enter the 'down' phase.
    When it rises back above up_threshold from the 'down' phase, one full rep is counted.
    """
    state = "up"
    reps = 0

    for angle in angle_series:
        if np.isnan(angle):
            continue

        if state == "up" and angle < down_threshold:
            state = "down"
        elif state == "down" and angle > up_threshold:
            state = "up"
            reps += 1

    return reps


rep_count = count_repetitions(df["right_knee_angle"])
print("Estimated repetitions (right knee):", rep_count)

## 32. Quality Checks

A general check on data quality: descriptive statistics and the count of missing (`NaN`)
values per column. Geometrically, no angle should ever fall outside the range of 0 to 180
degrees, since `calculate_angle` always returns a value within that range (or `NaN`).

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
angle_columns = [c for c in df.columns if c.endswith("_angle")]

out_of_range = {}
for col in angle_columns:
    valid = df[col].dropna()
    invalid = valid[(valid < 0) | (valid > 180)]
    out_of_range[col] = len(invalid)

print("Out-of-range values per column (should all be 0):")
for col, count in out_of_range.items():
    print(f"  {col}: {count}")

## 33. Limitations: The Effect of Camera View

The accuracy of the computed angles depends heavily on the filming conditions. The most
influential factors include:

- Camera distance
- Camera angle relative to the body
- Perspective (a 3D scene projected onto a 2D screen)
- Lighting
- Image quality (resolution, compression)
- Occlusion (part of the body hidden by an object or another body part)
- Clothing
- Body orientation (rotation relative to the camera)
- Speed of movement (motion blur)

For this reason, the output of this project is a **computer-vision movement analysis** and
**should not** be used as a definitive medical diagnostic tool.

## 34. A Note on 2D vs. 3D

In this version, all angles are computed based only on `[x, y]` (two dimensions). Each
landmark also has a `z` value (relative depth), and a 3D version of the angle calculation
could be developed in the future using `[x, y, z]`.

However, the `z` value in MediaPipe is a **relative estimate**, not a precise depth
measurement on par with medical-grade equipment (e.g. MRI or lab-grade motion capture),
and should not be assumed equivalent to that.

## 35. Troubleshooting (Common Errors)

| Problem | Likely cause | Fix |
|---|---|---|
| `FileNotFoundError: Model file not found` | The `.task` file is not at the expected path | Download the model file and check `MODEL_PATH` |
| `FileNotFoundError: Video file not found` | `VIDEO_PATH` is wrong | Fix the video path in the Configuration cell |
| `RuntimeError: Could not open video` | Unsupported video format or corrupted file | Test the file in another player, or convert it to mp4/H.264 |
| `No pose detected` in most frames | The person isn't fully in frame, low light, camera too far/close | Adjust camera angle, distance, or lighting |
| Many `NaN` values in some columns | Low visibility for that landmark in many frames | Lower `VISIBILITY_THRESHOLD`, or adjust camera angle (carefully) |
| Angles jump around / noisy | Natural per-frame detection noise | Use the `_smooth` columns (section 29) |

## 36. Final Reusable Pipeline — Summary

This entire notebook ultimately revolves around a single, reusable function:

```python
df = process_video(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_csv=OUTPUT_CSV_PATH,
    output_video=OUTPUT_VIDEO_PATH,
    show=False,
)
```

This is exactly what was run in section 25, with the result stored in `df` — from opening
the video, through pose detection, computing all joint angles, drawing annotations, saving
the annotated video, and building/saving the CSV.

### Summary of the Project's Core Functions

| Function | Role |
|---|---|
| `resize_for_display()` | Resizes the display copy of an image (without altering the original frame) |
| `landmark_to_pixel()` | Converts normalized coordinates to pixels |
| `get_landmark()` | Accesses a landmark by name |
| `is_landmark_visible()` | Checks whether a landmark is reliable |
| `calculate_angle()` | Generic calculation of the angle between three points |
| `calculate_joint_angles()` | Computes all joint angles for one frame |
| `draw_point()` / `draw_connection()` / `draw_angle()` | Draws landmarks, connecting lines, and angle values |
| `draw_pose_annotations()` | Combines the drawing functions for the whole body in one frame |
| `process_video()` | Runs the complete pipeline on a video file |

### Possible Future Extensions

- Adding a **Hand Landmarker** for more detailed finger analysis
- Extending angle computation to 3D (`x, y, z`)
- Calibrating `count_repetitions()` for different movements and exercises
- Processing multiple people at once (`num_poses > 1`)